# CQD Graph -> ADLS Bronze (POC ingest)

Runs the same `cqd.ingest` logic as the command line, inside Fabric. Lists Graph call records for the lookback window and writes immutable snapshots to `bronze/callrecords` plus the `control` ledger. Reruns are safe: versions already in the ledger are skipped.

**One-time setup**
1. Upload the repo's `cqd/` folder to the attached Lakehouse at `Files/code/cqd` (`__init__.py`, `graph.py`, `storage.py`, `probe.py`, `ingest.py`).
2. Store the SPN client secret in Azure Key Vault and give the notebook's running identity `Key Vault Secrets User`.
3. Set the parameters cell below. Never paste the secret into the notebook.

Schedule this notebook, then `cqd_bronze_to_silver_gold`, in a Data Pipeline (ingest first, on success run transform).

POC only. The enterprise design (Functions, managed identity, Service Bus) replaces this after approval.

In [ ]:
# Placeholders only. Set real values in the Fabric copy or pass them as pipeline notebook parameters; do not commit them.
TENANT_ID = "<tenant-id>"
CLIENT_ID = "<spn-client-id>"
STORAGE_ACCOUNT = "<storage-account>"
LOOKBACK_HOURS = 24
LIMIT = None  # set an integer to cap records per run
KEY_VAULT_URL = "https://<vault-name>.vault.usgovcloudapi.net/"
SECRET_NAME = "<key-vault-secret-name>"
CODE_DIR = "/lakehouse/default/Files/code"

In [ ]:
import importlib
import sys

for name in ("TENANT_ID", "CLIENT_ID", "STORAGE_ACCOUNT", "KEY_VAULT_URL", "SECRET_NAME"):
    if "<" in globals()[name]:
        raise RuntimeError(f"{name} is still a placeholder; set it in the parameters cell or the pipeline parameters")

for package in ("azure.identity", "azure.storage.blob", "requests"):
    try:
        importlib.import_module(package)
    except ImportError as exc:
        raise RuntimeError(f"{package} is missing; run: %pip install azure-identity azure-storage-blob requests") from exc

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from azure.identity import AzureAuthorityHosts, ClientSecretCredential
from azure.storage.blob import BlobServiceClient

from cqd.graph import GRAPH_SCOPE, GraphClient
from cqd.ingest import ingest
from cqd.storage import RawStore

In [ ]:
import requests

# notebookutils.getSecret only accepts vault.azure.net, so call the Gov vault REST API directly.
vault_token = notebookutils.credentials.getToken("https://vault.usgovcloudapi.net")
vault_response = requests.get(
    f"{KEY_VAULT_URL.rstrip('/')}/secrets/{SECRET_NAME}",
    params={"api-version": "7.4"},
    headers={"Authorization": f"Bearer {vault_token}"},
    timeout=(10, 30),
    allow_redirects=False,
)
del vault_token
if vault_response.status_code != 200:
    raise RuntimeError(f"Key Vault read failed: HTTP {vault_response.status_code}")
secret = vault_response.json()["value"]
del vault_response

with ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=secret,
    authority=AzureAuthorityHosts.AZURE_GOVERNMENT,
) as credential:
    graph = GraphClient(lambda: credential.get_token(GRAPH_SCOPE).token)
    service = BlobServiceClient(
        f"https://{STORAGE_ACCOUNT}.blob.core.usgovcloudapi.net", credential=credential
    )
    counts = ingest(graph, RawStore(service, TENANT_ID), LOOKBACK_HOURS, LIMIT)

del secret
print(f"Listed {counts['listed']}; stored {counts['stored']}; already present {counts['skipped']}.")